# U.S. Airline Operational Performance Analysis

**Author**: Ashwin Parthasarathy

**Dataset**: U.S. Bureau of Transportation Statistics - Reporting Carrier On-Time Performance

This project builds a reproducible Python data workflow to analyze recent U.S. airline operational performance. The analysis focuses on flight delays, cancellations, airports, carriers, routes, and temporal patterns using recent BTS flight operations data.

## 1. Setup
---

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
pd.set_option("display.max_columns", None)
print("Numpy: ", np.__version__)
print("Pandas: ", pd.__version__)

Numpy:  2.5.3
Pandas:  3.0.6


## 2. Data Ingestion
---

### Dataset Source and Scope

This analysis uses the U.S. Bureau of Transportation Statistics (BTS)
Reporting Carrier On-Time Performance dataset.

The working dataset covers January 2025 through July 2026. Raw monthly
BTS ZIP archives are downloaded reproducibly and converted to focused
Parquet files containing the operational fields required for this analysis.

The raw and processed datasets are excluded from Git because of their size.
The repository instead includes scripts to reproduce the local dataset.

In [9]:
PROJECT_ROOT = Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

parquet_files = sorted(PROCESSED_DATA_DIR.glob("bts_flights_*.parquet"))

print(f"Monthly files found: {len(parquet_files)}")
print(f"First file: {parquet_files[0].name}")
print(f"Last file:  {parquet_files[-1].name}")

Monthly files found: 19
First file: bts_flights_2025_01.parquet
Last file:  bts_flights_2026_07.parquet


In [11]:
# Take inventory of all files and inspect for data sufficiency (no. of records, start date, end date)
monthly_inventory = []

for path in parquet_files:
    dates = pd.read_parquet(
        path,
        columns=["FlightDate"],
    )

    monthly_inventory.append(
        {
            "file": path.name,
            "rows": len(dates),
            "start_date": dates["FlightDate"].min(),
            "end_date": dates["FlightDate"].max(),
        }
    )

inventory_df = pd.DataFrame(monthly_inventory)
inventory_df

,file,rows,start_date,end_date
0,bts_flights_2025_01.parquet,539747,2025-01-01,2025-01-31
1,bts_flights_2025_02.parquet,504884,2025-02-01,2025-02-28
2,bts_flights_2025_03.parquet,600872,2025-03-01,2025-03-31
3,bts_flights_2025_04.parquet,583950,2025-04-01,2025-04-30
4,bts_flights_2025_05.parquet,605648,2025-05-01,2025-05-31
5,bts_flights_2025_06.parquet,611575,2025-06-01,2025-06-30
6,bts_flights_2025_07.parquet,631428,2025-07-01,2025-07-31
7,bts_flights_2025_08.parquet,602378,2025-08-01,2025-08-31
8,bts_flights_2025_09.parquet,562439,2025-09-01,2025-09-30
9,bts_flights_2025_10.parquet,605844,2025-10-01,2025-10-31


In [12]:
print(f"Total rows: {inventory_df['rows'].sum():,}")
print("Date range: ", inventory_df['start_date'].min(), " to ", inventory_df['end_date'].max())

Total rows: 11,121,962
Date range:  2025-01-01 00:00:00  to  2026-07-31 00:00:00


In [13]:
# Sample one month's data
sample_df = pd.read_parquet(parquet_files[0])

print(f"Shape: {sample_df.shape}")
sample_df.head()

Shape: (539747, 41)


,Year,Month,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,Flight_Number_Reporting_Airline,OriginAirportID,Origin,OriginCityName,OriginState,DestAirportID,Dest,DestCityName,DestState,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2025,1,3,2025-01-01,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,659,656.0,-3.0,0.0,0.0,0600-0659,23.0,9.0,1020,1013.0,-7.0,0.0,0.0,1000-1059,0.0,NaN,0.0,381.0,377.0,345.0,2475.0,NaN,NaN,NaN,NaN,NaN
1,2025,1,4,2025-01-02,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,659,652.0,-7.0,0.0,0.0,0600-0659,30.0,7.0,1020,1022.0,2.0,2.0,0.0,1000-1059,0.0,NaN,0.0,381.0,390.0,353.0,2475.0,NaN,NaN,NaN,NaN,NaN
2,2025,1,5,2025-01-03,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,659,652.0,-7.0,0.0,0.0,0600-0659,13.0,11.0,1020,1003.0,-17.0,0.0,0.0,1000-1059,0.0,NaN,0.0,381.0,371.0,347.0,2475.0,NaN,NaN,NaN,NaN,NaN
3,2025,1,6,2025-01-04,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,700,653.0,-7.0,0.0,0.0,0700-0759,24.0,10.0,1026,1016.0,-10.0,0.0,0.0,1000-1059,0.0,NaN,0.0,386.0,383.0,349.0,2475.0,NaN,NaN,NaN,NaN,NaN
4,2025,1,7,2025-01-05,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,659,655.0,-4.0,0.0,0.0,0600-0659,26.0,5.0,1020,1013.0,-7.0,0.0,0.0,1000-1059,0.0,NaN,0.0,381.0,378.0,347.0,2475.0,NaN,NaN,NaN,NaN,NaN


In [14]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 539747 entries, 0 to 539746
Data columns (total 41 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   Year                             539747 non-null  int64         
 1   Month                            539747 non-null  int64         
 2   DayOfWeek                        539747 non-null  int64         
 3   FlightDate                       539747 non-null  datetime64[us]
 4   Reporting_Airline                539747 non-null  str           
 5   DOT_ID_Reporting_Airline         539747 non-null  int64         
 6   Flight_Number_Reporting_Airline  539747 non-null  int64         
 7   OriginAirportID                  539747 non-null  int64         
 8   Origin                           539747 non-null  str           
 9   OriginCityName                   539747 non-null  str           
 10  OriginState                      539747 non-null  str  

In [18]:
# Summary of dataset

print(f"Monthly files: {len(parquet_files)}")
print(f"Total rows: {inventory_df['rows'].sum()}")
print(f"Date range: {inventory_df['start_date'].min()} to {inventory_df['end_date'].max()}")

Monthly files: 19
Total rows: 11121962
Date range: 2025-01-01 00:00:00 to 2026-07-31 00:00:00


> The working dataset contains 11.12M flight records across 19 monthly partitions. To keep the workflow reproducible and memory-efficient, the analysis processes monthly parquet partitions rather than loading the entire dataset into memory at once.

## 3. Data Cleaning
---

## 4. Exploratory Data Analysis
---

## 5. Visualizations
---

## 6. Summary & Interpretation
---